In [1]:
### Timing decorator
import time

def timing(func):
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        end = time.perf_counter()
        print(f"{func.__name__} took {end - start:.4f} seconds")
        return result
    return wrapper


In [2]:
# Enable autoreload of imported modules

%load_ext autoreload
%autoreload 2

In [3]:
import pandas as pd
from pathlib import Path
import yaml
import spacy
@timing
def load_config(path):
    """
    Load event config file and data.

    Returns: Config `cfg` and DataFrame `df`.
    """
    config_file = path
    cfg = yaml.safe_load(Path(config_file).read_text(encoding='utf-8')).get('event', {})

    # Normalize keywords
    nlp = spacy.load("en_core_web_lg")
    doc = nlp(" ".join(cfg["keywords"]))
    cfg['keywords'] = [t.lemma_ for t in doc]

    df = pd.read_csv(cfg['input_file'])

    return cfg, df

In [4]:
from preprocessing.normalize_text import normalize_text_series

@timing
def normalize_texts(df, batch_size=50, n_process=10):
    """
    Create clean text columns for 'title', 'text' and 'first_para'.
    """

    if 'title' in df:
        df['title_clean'] = normalize_text_series(df['title'], 
                                                  batch_size=batch_size, 
                                                  n_process=n_process)
    if 'text' in df:
        df['text_clean'] = normalize_text_series(df['text'], 
                                                  batch_size=batch_size, 
                                                  n_process=n_process)
    if 'first_para' in df:
        df['first_para_clean'] = normalize_text_series(df['first_para'], 
                                                  batch_size=batch_size, 
                                                  n_process=n_process)
    
    return df

In [5]:
from preprocessing.tfidf_dedupe import dedupe_tfidf_cosine

@timing
def dedupe_texts(df, text_col):
    """
    Dedupe entries in `df` with nearly identical values in `text_col`. Keeps the longest text by default.
    """
    threshold = 0.9
    min_df = 1  # min doc frequency
    ngram_min = 1
    ngram_max = 2
    max_features=None
    prefer_longer = True
    block_by_length = True  # compare only to similar length documents
    return_groups = True

    deduped_df, report = dedupe_tfidf_cosine(
        df,
        text_col=text_col,
        threshold=threshold,
        min_df=min_df,
        ngram_min=ngram_min,
        ngram_max=ngram_max,
        max_features=max_features,
        prefer_longer=prefer_longer,
        block_by_length=block_by_length,
        return_groups=return_groups,
    )

    return deduped_df, report

In [ ]:
# Run text cleanup for all configs
from pathlib import Path
import pandas as pd
import yaml

config_path = Path("./config/")
config_files = [f for f in config_path.iterdir() if f.suffix=='.yaml']

for f in config_files:
    cfg = yaml.safe_load(Path(f).read_text(encoding='utf-8')).get('event', {})
    print(cfg)
    if not Path(cfg['input_file']).exists():
        print(f"{cfg['input_file']} not found")
        continue
    output_file = Path(str(cfg['input_file']).replace("/raw/", "/clean/"))
    if output_file.exists():
        print(f"Already exists: {output_file}")
        continue

    df = pd.read_csv(cfg['input_file'])
    df = normalize_texts(df)
    df, report_text = dedupe_texts(df, text_col='text_clean')
    df, report_title = dedupe_texts(df, text_col='title_clean')
    df, report_para = dedupe_texts(df, text_col='first_para')

    print(f"Documents after deduping: {len(df)}")
    print(f"Saving dataframe to: {output_file}")
    df.to_csv(output_file, index=None)

{'name': '2025_floods_TX', 'state': 'TX', 'input_file': 'data/event_data/raw/2025_floods_TX.csv', 'start_date': datetime.date(2025, 6, 27), 'end_date': datetime.date(2025, 8, 4), 'onset_date': datetime.date(2025, 7, 4), 'keywords': ['flooding', 'rain', 'storms', 'rivers', 'evacuation', 'damage', 'emergency', 'texas', 'disaster', 'response']}
Already exists: data/event_data/clean/2025_floods_TX.csv
{'name': '2022_uvalde_tx', 'state': 'TX', 'input_file': 'data/event_data/raw/2022_uvalde_tx.csv', 'start_date': datetime.date(2022, 5, 17), 'end_date': datetime.date(2022, 6, 24), 'onset_date': datetime.date(2022, 5, 24), 'keywords': ['shooting', 'school', 'gunman', 'victims', 'children', 'gun violence', 'gun', 'kill', 'mass shooting', 'fatalities']}
Already exists: data/event_data/clean/2022_uvalde_tx.csv
{'name': '2024_Milton_FL', 'state': 'FL', 'input_file': 'data/event_data/raw/2024_Milton_FL.csv', 'start_date': datetime.date(2024, 10, 1), 'end_date': datetime.date(2024, 11, 10), 'keyword